# Taxonomic and Distribution Data on African Pipistrelloid Bats and Sokoke Scops Owl Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the "Taxonomic and Distribution Data on African Pipistrelloid Bats and Sokoke Scops Owl from Central Angolan Highlands and Africa" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
[https://sen.science/doi/10.71728/senscience.wxag-kz12/fair2.json](https://sen.science/doi/10.71728/senscience.wxag-kz12/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will display the dataset title and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.wxag-kz12/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This is important to know what tables (RecordSets) and fields (columns) can be accessed for analysis.

The following code lists all record sets in the dataset and their fields, referencing entities by their `@id`. (Replace `<id_of_the_record_set>` with actual `@id` as needed in the next sections.)

In [ ]:
# Explore the record sets available in the dataset
record_sets = [rs for rs in getattr(metadata, 'record_sets', [])]
if not record_sets:
    # Try alternative (as per Croissant 1.0, likely as `metadata.recordSet`):
    record_sets = getattr(metadata, 'recordSet', [])

print(f"Found {len(record_sets)} record set(s) in the dataset.")
for rs in record_sets:
    this_rs = dataset.metadata.by_id(rs['@id']) if isinstance(rs, dict) and '@id' in rs else rs
    print(f"\nRecord Set @id: {getattr(this_rs,'@id', '')}")
    print(f"  Name: {getattr(this_rs,'name','')}")
    print(f"  Description: {getattr(this_rs,'description','')}")
    fields = getattr(this_rs, 'fields', []) or getattr(this_rs, 'field', [])  # Croissant 1.0 & 1.1
    print(f"  Fields in this Record Set:")
    for f in fields:
        field = dataset.metadata.by_id(f['@id']) if isinstance(f, dict) and '@id' in f else f
        print(f"    Field @id: {getattr(field,'@id','')} Name: {getattr(field,'name','')}")

# If the record sets list is empty, print as much info as possible
if not record_sets:
    print("No explicit record sets found in the Croissant metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references are by their `@id` (record set and field `@id`).

**NOTE:** If you have identified desired record set `@id` from above, update `selected_record_set_id` and (optionally) limit fields.

In [ ]:
# ====== User input section - fill these after running section above =======
# Update with a valid record set `@id` shown above.
selected_record_set_id = None
# Optionally, limit to certain fields (by their @id)
selected_field_ids = None  # e.g., ['cr:Field:species', ...]
# ======================================================================

# Helper to extract records by record set @id
dataframes = {}
if selected_record_set_id is not None:
    print(f"Loading records for record set: {selected_record_set_id}")
    records = list(dataset.records(record_set=selected_record_set_id))
    df = pd.DataFrame(records)
    if selected_field_ids:
        fields = [f for f in df.columns if f in selected_field_ids]
        df = df[fields]
    dataframes[selected_record_set_id] = df
    print(f"DataFrame columns: {df.columns.tolist()}")
    display(df.head())
else:
    print("Please set `selected_record_set_id` in the cell above to continue.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping. All fields referenced by `@id`.

Example: Filter numeric field, normalize, group by another field.

In [ ]:
# ====== User input section - set below as per your schema/fields =======
# Example numeric field and group field @id from fields printed above (e.g., 'cr:Field:year_collected')
numeric_field_id = None  # e.g., 'cr:Field:year_collected'
group_field_id = None    # e.g., 'cr:Field:species', optional
filter_threshold = None  # e.g., 2000
record_set_id = selected_record_set_id
# ======================================================================

if record_set_id is not None and record_set_id in dataframes:
    df = dataframes[record_set_id]
    if numeric_field_id is not None and numeric_field_id in df.columns:
        thresh = filter_threshold if filter_threshold is not None else 0
        filtered_df = df[df[numeric_field_id] > thresh].copy()
        print(f"Filtered records with {numeric_field_id} > {thresh}:\n", filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized column for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

        if group_field_id is not None and group_field_id in df.columns:
            grouped_df = (
                filtered_df.groupby(group_field_id)[numeric_field_id]
                .mean().reset_index()
                .rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
            )
            print(f"\nGrouped means by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("Please set `numeric_field_id` to a numeric field present in your DataFrame.")
else:
    print("Please set both `selected_record_set_id` (above) and ensure its DataFrame loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields. Below is a template for creating a histogram or a boxplot using the selected field IDs. Update the field IDs as needed. Visualizations are based on DataFrame fields using their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id is not None and record_set_id in dataframes:
    df = dataframes[record_set_id]

    # Example: Histogram of numeric field
    if numeric_field_id is not None and numeric_field_id in df.columns:
        plt.figure(figsize=(7,4))
        sns.histplot(df[numeric_field_id], bins=30, kde=True)
        plt.title(f"Histogram of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()

    # Example: Boxplot by group
    if group_field_id is not None and group_field_id in df.columns and numeric_field_id is not None and numeric_field_id in df.columns:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=90)
        plt.show()
else:
    print("Please run previous cells and set valid field IDs.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration. This notebook demonstrated how to use the `mlcroissant` library to inspect Croissant-based datasets, using only valid `@id` references for all entities (record sets and fields). Further exploration can make use of additional record sets and fields as needed per the Croissant schema.